# Агентные LLM: вызов инструментов и контроль формата ответа

В предыдущих семинарах мы разбирали:
- **Instruct Fine-tuning** — как научить LLM следовать инструкциям (SFT, LoRA/QLoRA)
- **RLHF / DPO / GRPO** — как выровнять модель по человеческим предпочтениям или верифицируемым наградам

Сегодня поговорим про две связанные проблемы, которая стала очевидны как только появились модели хорошо обученные и использованием этих инструментов. 

Во-первых, LLM это все еще LLM и даже с очень хорошими обучающими данными, все равно есть вещи, которые делать очень сложно просто в силу архитектуры модели. Классический пример - это спросить модель сколько букв r в слове strawberry или сравнить 9.9 и 9.10 (первую сложно решить потому что модель работает с токенами, а вторую потому что модель должна выучить математику из выборки и если задуматься то это очень сложно). И проблема тут именно в LLM, сами по себе это тривиальные задачи. И чтобы не биться головой о стены придумали tool calling - это спобоб для модели использовать "инструменты", которые могут закрывать ей такие пробелы. Для математического примера можно просто обратиться к калькулятору, а для strawberry подойдут какие-то текстовые инструменты вроде регулярок. И соответственно вместо того, чтобы учить модель решать задачи напрямую, можно учить ее использовать правильные инструменты. Эта идея оказалась очень хорошей и масштабируемой. И даже такие фундаментальные проблемы как обучение модели новым знаниям частично можно решить за счет tool (если модель умеет искать в интернете, то по сути ее можно не переобучать новым фактам!)

Во-вторых, помимо простых разговоров в чате, модели хотелось бы использовать для автоматизации процессов (например, для модерации). Но в этом случае мы должны быть уверены в том, что модель ответит в нужном формате, чтобы автоматически распарсить ответ. А так как модель стохастическая, то сделать это очень сложно. Поэтому стали развиваться подходы, которые в общем можно назвать Structured Outputs. Когда мы имеем полный доступ к модели и можем регулировать sampling мы можем использовать разного рода грамматики для проверки ответов на лету и выбора только подходящих токенов. А когда такого доступа нет, нам приходится полагаться на уже ставший стандартом подход, где модель дообучена следовать формату заданному какой-то стандартной схемой. Все лучшие модели так или иначе это поддеживают и открытые модели тоже постепенно догоняют.

Но прежде чем мы посмотрим на это на практике, сделаем небольшое отступление и поговорим про агентность! Две эти идеи это наверное основные элементы, которые позволили сделать из chat ботов, настоящих автономных (до определенной степени конечно!) ассистенов, которые взаимодействуют со средой, планируют, запоминают, могут совершать множество отдельных действий без прямой инструкции и т.п.

Во многом текущие LLM основаны не фреймворке, который называется ReAct: Reason + Act.
Основополагающая статья: [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) (Yao et al., ICLR 2023).

Идея проста: **чередовать рассуждения (Reason) и действия (Action)**. Модель:
1. Думает, какое действие нужно сделать
2. Вызывает инструмент (поиск, калькулятор, API)
3. Получает результат (Observation)
4. Снова думает, что делать дальше
5. Повторяет, пока не придёт к ответу

```
Question: What is the elevation range for the area that the eastern sector of the
Colorado orogeny extends into?

Thought: I need to search Colorado orogeny, find the area that the eastern sector
of the Colorado orogeny extends into, then find the elevation range of the area.
Action: Search[Colorado orogeny]
Observation: The Colorado orogeny was an episode of mountain building...

Thought: I need to search "High Plains" for its elevation range.
Action: Search[High Plains]
Observation: The High Plains are a subregion of the Great Plains...

Thought: The elevation range is 1,800 to 7,000 ft.
Action: Finish[1,800 to 7,000 ft]
```


Для рассуждений модель можно использовать просто токены или специальные <thinking> токены, а вот для Act шага нужно, чтобы вызовы инструментов срабатывали (т.е. имели правильный формат). 

---

## Форматы вызова инструментов

Прежде чем перейти к коду — разберёмся с форматами. Это важно, потому что разные модели и библиотеки используют разные (но похожие) форматы.

### OpenAI Function Calling формат

OpenAI ввела стандарт, который стал де-факто индустриальным. Вызов инструмента — это часть сообщения с ролью `assistant`.

**Описание инструмента (передаётся в `tools`):**
```json
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get current weather for a location",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string",
          "description": "City name, e.g. 'Moscow'"
        },
        "unit": {
          "type": "string",
          "enum": ["celsius", "fahrenheit"]
        }
      },
      "required": ["location"]
    }
  }
}
```

**Ответ модели с вызовом инструмента:**
```json
{
  "role": "assistant",
  "content": null,
  "tool_calls": [
    {
      "id": "call_abc123",
      "type": "function",
      "function": {
        "name": "get_weather",
        "arguments": "{\"location\": \"Moscow\", \"unit\": \"celsius\"}"
      }
    }
  ]
}
```

Обратите внимание: `arguments` — это **строка** с JSON, а не сам JSON-объект.

**Ответ инструмента (сообщение с ролью `tool`):**
```json
{
  "role": "tool",
  "tool_call_id": "call_abc123",
  "content": "{\"temperature\": -3, \"weather\": \"cloudy\"}"
}
```

---

### HuggingFace Chat Templates

HuggingFace унифицировал этот формат через [chat templates](https://huggingface.co/docs/transformers/en/chat_templating). Ключевое отличие:

- Разные модели (Qwen, Llama, Mistral) преобразуют это в свой внутренний формат через jinja2-шаблон

Это позволяет писать **один код для всех моделей**:
```python
messages = [
    {"role": "user", "content": "Какая погода в Москве?"},
    {"role": "assistant", "tool_calls": [
        {"type": "function", "function": {"name": "get_weather", "arguments": {"location": "Moscow"}}}
    ]},
    {"role": "tool", "name": "get_weather", "content": "{\"temp\": -3}"}
]
tokenizer.apply_chat_template(messages, tools=[...], tokenize=False)
```

---

### Как это выглядит «изнутри» для разных моделей

Каждая модель-семейство имеет свой токенизированный формат:

**Qwen2.5/Qwen3** использует специальные теги:
```
<|im_start|>system
You are a helpful assistant.

# Tools
You may call one or more functions to assist with the user query.

{"name": "get_weather", "description": "...", "parameters": {...}}<|im_end|>
<|im_start|>user
Какая погода в Москве?<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "get_weather", "arguments": {"location": "Moscow"}}
</tool_call><|im_end|>
```

**Llama 3.1+** использует специальные токены `<|python_tag|>` и `<|eot_id|>`.

**Mistral V2+** использует control tokens: `[AVAILABLE_TOOLS]`, `[TOOL_CALLS]`, `[TOOL_RESULTS]`.

За это и нужны chat templates — они скрывают эти различия.

## Практика: вызов инструментов с open models

In [87]:
%pip install -q transformers accelerate bitsandbytes

In [91]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Используем Qwen3-0.6B — маленькая, но нативно поддерживает tool calling
MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print(f"Модель загружена: {MODEL_NAME}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Модель загружена: Qwen/Qwen3-0.6B


### Определяем инструменты

В HuggingFace transformers ≥ 4.40 инструменты можно описывать как Python-функции — библиотека сама построит JSON Schema.

In [92]:
def get_weather(location: str, unit: str = "celsius") -> dict:
    """
    Get current weather information for a location.
    
    Args:
        location: City name, e.g. 'Moscow' or 'New York'
        unit: Temperature unit, either 'celsius' or 'fahrenheit'
    """
    # В реальном приложении здесь был бы API-запрос
    # Для демо возвращаем заглушку
    return {"temperature": -3, "weather": "cloudy", "unit": unit}


def search_web(query: str, max_results: int = 3) -> list:
    """
    Search the web for information.
    
    Args:
        query: Search query string
        max_results: Maximum number of results to return
    """
    # Заглушка
    return [{"title": f"Result {i}", "snippet": f"..."} for i in range(max_results)]


def calculate(expression: str) -> float:
    """
    Evaluate a mathematical expression.
    
    Args:
        expression: Mathematical expression as a string, e.g. '2 + 3 * 4'
    """
    # ВАЖНО: eval очень опасная функция и не вызывайте ее на всем подряд!!
    import ast
    return float(eval(expression, {"__builtins__": {}}, {}))


tools = [get_weather, search_web, calculate]
print("Инструменты определены:", [t.__name__ for t in tools])

Инструменты определены: ['get_weather', 'search_web', 'calculate']


### Посмотрим, во что превратится описание инструмента

Tokenizer.apply_chat_template использует `get_json_schema()` для конвертации Python-функций в JSON Schema.

In [93]:
from transformers.utils import get_json_schema

# Посмотрим на автоматически сгенерированную JSON Schema
schema = get_json_schema(get_weather)
print(json.dumps(schema, indent=2, ensure_ascii=False))

{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get current weather information for a location.",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string",
          "description": "City name, e.g. 'Moscow' or 'New York'"
        },
        "unit": {
          "type": "string",
          "description": "Temperature unit, either 'celsius' or 'fahrenheit'"
        }
      },
      "required": [
        "location"
      ]
    },
    "return": {
      "type": "object"
    }
  }
}


In [94]:
# Посмотрим как выглядит промпт с инструментами в нотации Qwen3
messages = [
    {"role": "user", "content": "Какая сейчас погода в Москве?"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "Get current weather information for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "City name, e.g. 'Moscow' or 'New York'"}, "unit": {"type": "string", "description": "Temperature unit, either 'celsius' or 'fahrenheit'"}}, "required": ["location"]}, "return": {"type": "object"}}}
{"type": "function", "function": {"name": "search_web", "description": "Search the web for information.", "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Search query string"}, "max_results": {"type": "integer", "description": "Maximum number of results to return"}}, "required": ["query"]}, "return": {"type": "object"}}}
{"type": "function", "function": {"name

In [95]:
def generate_response(messages, tools=None, max_new_tokens=512):
    """Генерируем ответ модели (с поддержкой инструментов или без)."""
    input_text = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Декодируем только новые токены
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    return response_text


messages = [{"role": "user", "content": "Какая сейчас погода в Москве?"}]
response = generate_response(messages, tools=tools)
print("Ответ модели:")
print(response)

Ответ модели:
<think>
Okay, the user is asking about the current weather in Moscow. Let me check the tools available. There's a function called get_weather that requires the location and optionally the unit. The user didn't specify the unit, so I'll default to Celsius. I need to make sure the location is provided, which it is. So I should call get_weather with location "Moscow" and unit "celsius". That should give the user the information they need.
</think>

<tool_call>
{"name": "get_weather", "arguments": {"location": "Moscow"}}
</tool_call>


### Парсим вызов инструмента и выполняем его

In [96]:
def parse_tool_call(response_text):
    """Парсим ответ модели и извлекаем вызов инструмента.
    
    Qwen3 оборачивает вызов в <tool_call>...</tool_call>.
    Разные модели используют разные маркеры — в реальных проектах
    лучше использовать tokenizer.parse_tool_call() если доступно.
    """
    import re
    
    # Ищем блок <tool_call>...</tool_call>
    match = re.search(r'<tool_call>(.*?)</tool_call>', response_text, re.DOTALL)
    if match:
        tool_call_json = match.group(1).strip()
        return json.loads(tool_call_json)
    
    # Fallback: пробуем найти JSON напрямую
    try:
        return json.loads(response_text)
    except:
        return None


tool_registry = {
    "get_weather": get_weather,
    "search_web": search_web,
    "calculate": calculate,
}

def execute_tool_call(tool_call):
    """Выполняем вызов инструмента."""
    name = tool_call["name"]
    arguments = tool_call.get("arguments", {})
    
    if name not in tool_registry:
        return {"error": f"Unknown tool: {name}"}
    
    return tool_registry[name](**arguments)


# Разбираем ответ модели
messages = [{"role": "user", "content": "Какая сейчас погода в Москве?"}]
response = generate_response(messages, tools=tools)

tool_call = parse_tool_call(response)
print("Вызов инструмента:", tool_call)

if tool_call:
    result = execute_tool_call(tool_call)
    print("Результат инструмента:", result)

Вызов инструмента: {'name': 'get_weather', 'arguments': {'location': 'Moscow'}}
Результат инструмента: {'temperature': -3, 'weather': 'cloudy', 'unit': 'celsius'}


### Полный цикл с Reason и Act

In [97]:
def run_agent(user_query, tools, max_turns=5, verbose=True):
    """Простой агент с циклом Thought → Action → Observation."""
    
    messages = [{"role": "user", "content": user_query}]
    tool_map = {t.__name__: t for t in tools}
    
    for turn in range(max_turns):
        if verbose:
            print(f"\n--- Turn {turn + 1} ---")
        
        response = generate_response(messages, tools=tools)
        
        if verbose:
            print(f"Модель: {response[:200]}")
        
        # Пробуем разобрать вызов инструмента
        tool_call = parse_tool_call(response)
        
        if tool_call is None:
            # Нет вызова инструмента — это финальный ответ
            if verbose:
                print("\n=== Финальный ответ ===")
                print(response)
            return response, messages
        
        # Добавляем ответ ассистента с tool_call в историю
        messages.append({
            "role": "assistant",
            "tool_calls": [{
                "type": "function",
                "function": {
                    "name": tool_call["name"],
                    "arguments": tool_call.get("arguments", {})
                }
            }]
        })
        
        # Выполняем инструмент
        tool_name = tool_call["name"]
        if tool_name in tool_map:
            result = tool_map[tool_name](**tool_call.get("arguments", {}))
        else:
            result = {"error": f"Инструмент не найден: {tool_name}"}
        
        if verbose:
            print(f"Инструмент {tool_name} → {result}")
        
        # Добавляем результат инструмента
        messages.append({
            "role": "tool",
            "name": tool_name,
            "content": json.dumps(result, ensure_ascii=False)
        })
    
    return None, messages


# Тестируем агента
final_answer, conversation = run_agent(
    "Сколько будет 15 * 7 + 42?",
    tools=[calculate],
    verbose=True
)


--- Turn 1 ---
Модель: <think>
Okay, the user is asking how much 15 multiplied by 7 plus 42 is. Let me think. First, I need to calculate 15 times 7. That should be 105. Then, adding 42 to that result. So 105 plus 42 equals 
Инструмент calculate → 147.0

--- Turn 2 ---
Модель: <think>
Okay, let's see. The user asked, "Сколько будет 15 * 7 + 42?" which translates to "How much will 15 multiplied by 7 plus 42 be?" So I need to calculate that expression.

First, I remember that

=== Финальный ответ ===
<think>
Okay, let's see. The user asked, "Сколько будет 15 * 7 + 42?" which translates to "How much will 15 multiplied by 7 plus 42 be?" So I need to calculate that expression.

First, I remember that the order of operations is parentheses, exponents, multiplication and division, and then addition and subtraction. Since there are no parentheses here, I should handle multiplication and addition from left to right. 

So, 15 multiplied by 7 is 105. Then adding 42 to that gives 105 + 42. Let me 

### Параллельные вызовы инструментов

Современные модели поддерживают **parallel tool calling** — вызов нескольких инструментов за один шаг. Но это не значит что модель генерирует параллельно! Это все еще авторегресионная модель. Parallel tool calling это скорее про то как функции вызываются - можно либо останавливать модель и вызывать функцию, а затем передавать ответ в модель, чтобы получить следующее действие, а можно вызываться и генерировать параллельно и подавать результаты, когда они будут готовы (как в async программировании)

In [98]:
# Запрос, требующий нескольких инструментов
messages = [
    {"role": "user", "content": "Скажи погоду в Москве и Лондоне одновременно"}
]

response = generate_response(messages, tools=[get_weather])
print("Ответ модели (может быть несколько <tool_call>):")
print(response)

Ответ модели (может быть несколько <tool_call>):
<think>
Okay, the user is asking for the weather in Moscow and London at the same time. Let me check the tools available. The only tool provided is get_weather, which requires a location and optionally a unit. The function needs the location as a required parameter. Since the user wants both cities, I need to make two separate calls to get_weather for Moscow and London. Each call will have the respective location and maybe the same unit. I should structure two tool calls here. Let me confirm the parameters: location is required, so each call will include that. The unit isn't specified, so I can leave it out or use a default. Alright, I'll generate two tool calls for each city.
</think>

<tool_call>
{"name": "get_weather", "arguments": {"location": "Moscow"}}
</tool_call>
<tool_call>
{"name": "get_weather", "arguments": {"location": "London"}}
</tool_call>


---

### Smolagents: фреймворк для агентов от HuggingFace

[Smolagents](https://huggingface.co/docs/smolagents) — минималистичная библиотека (~1000 строк) для построения агентов на любых LLM.

Тут можно даже конвертировать питоновские функции в tools!

In [99]:
# %pip install "smolagents[toolkit]" -U

In [12]:
from smolagents import CodeAgent, ToolCallingAgent, InferenceClientModel, tool

# Определяем инструменты через декоратор
@tool
def get_temperature(city: str) -> str:
    """Returns the current temperature for a given city.
    
    Args:
        city: Name of the city to get temperature for.
    """
    # Заглушка
    temps = {"moscow": -3, "london": 8, "tokyo": 15}
    temp = temps.get(city.lower(), 20)
    return f"{temp}°C"


@tool  
def convert_celsius_to_fahrenheit(celsius: float) -> float:
    """Converts temperature from Celsius to Fahrenheit.
    
    Args:
        celsius: Temperature in Celsius degrees.
    """
    return celsius * 9/5 + 32


print("Инструменты для smolagents определены")

Инструменты для smolagents определены


In [13]:
# такую структуру можно часто встретить в LLM фреймворках
# часто под агентов понимают while loop с моделью у которой есть список инструментов
import torch
from smolagents import TransformersModel

local_model = TransformersModel(
    model_id="Qwen/Qwen3-0.6B",
    device_map="auto",
    torch_dtype=torch.bfloat16
)

agent = ToolCallingAgent(
    tools=[get_temperature, convert_celsius_to_fahrenheit],
    model=local_model
)

result = agent.run("What is the temperature in Moscow in Fahrenheit?")
print("Результат:", result)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the temperature in Moscow in Fahrenheit?                                                                │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3-0.6B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_temperature' with arguments: {'city': 'Moscow'}                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '26.6°F'}                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 26.6°F

Final answer: 26.6°F

[Step 2: Duration 5.58 seconds| Input tokens: 2,562 | Output tokens: 1,027]

Результат: 26.6°F


---

## Обучение open models вызову инструментов

Tool calling это такая же генерация токенов и обучение моделей происходит точно также как и во всех остальных случаях. Для начала можно обучить с помощью SFT, чтобы модель выучила формат. В каком-то смысле это еще одна абстрактная задача в дополнение к следованию инструкциям.

### Датасет: xlam-function-calling-60k

Статья: [xLAM: A Family of Large Action Models to Accelerate AI Agent Tasks](https://arxiv.org/abs/2409.03215) (Salesforce, 2024)

Датасет [Salesforce/xlam-function-calling-60k](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k) содержит 60,000 примеров вызовов функций с:
- Описанием инструментов (JSON Schema)
- Запросом пользователя
- Правильным вызовом (одиночным или параллельным)


In [100]:
# %pip install datasets

In [4]:
from datasets import load_dataset
import json

# Загружаем датасет
xlam_dataset = load_dataset("Salesforce/xlam-function-calling-60k", split="train", token=TOKEN)
print(f"Размер датасета: {len(xlam_dataset)}")
print(f"Колонки: {xlam_dataset.column_names}")
print()

# Смотрим на пример
example = xlam_dataset[0]
print("=== Пример ===")
print("Запрос:", example["query"][:200])
print()
tools = json.loads(example["tools"])
print(f"Инструментов: {len(tools)}")
print("Первый инструмент:", json.dumps(tools[0], indent=2, ensure_ascii=False)[:300])
print()
print("Правильные вызовы:", example["answers"])

Размер датасета: 60000
Колонки: ['id', 'query', 'answers', 'tools']

=== Пример ===
Запрос: Where can I find live giveaways for beta access and games?

Инструментов: 1
Первый инструмент: {
  "name": "live_giveaways_by_type",
  "description": "Retrieve live giveaways from the GamerPower API based on the specified type.",
  "parameters": {
    "type": {
      "description": "The type of giveaways to retrieve (e.g., game, loot, beta).",
      "type": "str",
      "default": "game"
    

Правильные вызовы: [{"name": "live_giveaways_by_type", "arguments": {"type": "beta"}}, {"name": "live_giveaways_by_type", "arguments": {"type": "game"}}]


In [5]:
# Посмотрим на разные типы вызовов: одиночный, параллельный
import json

single_calls = []
parallel_calls = []

for example in xlam_dataset.select(range(1000)):
    answers = json.loads(example["answers"]) if isinstance(example["answers"], str) else example["answers"]
    if len(answers) == 1:
        single_calls.append(example)
    elif len(answers) > 1:
        parallel_calls.append(example)

print(f"Одиночные вызовы: {len(single_calls)}")
print(f"Параллельные вызовы: {len(parallel_calls)}")

# Пример параллельного вызова
if parallel_calls:
    ex = parallel_calls[0]
    print("\n=== Параллельный вызов ===")
    print("Запрос:", ex["query"][:200])
    print("Вызовы:", ex["answers"])

Одиночные вызовы: 472
Параллельные вызовы: 528

=== Параллельный вызов ===
Запрос: Where can I find live giveaways for beta access and games?
Вызовы: [{"name": "live_giveaways_by_type", "arguments": {"type": "beta"}}, {"name": "live_giveaways_by_type", "arguments": {"type": "game"}}]


### SFT на данных с вызовами инструментов (TRL)

TRL ≥ 0.13 поддерживает обучение на данных в формате `conversations` + `tools` через `SFTTrainer`.

Структура примера для обучения:
```python
{
    "messages": [
        {"role": "user", "content": "Какая погода в Москве?"},
        {"role": "assistant", "tool_calls": [
            {"type": "function", "function": {"name": "get_weather", "arguments": {"location": "Moscow"}}}
        ]},
        {"role": "tool", "name": "get_weather", "content": "{\"temp\": -3}"},
        {"role": "assistant", "content": "В Москве сейчас -3°C."}
    ],
    "tools": [get_weather_schema]  # JSON Schema инструментов
}
```

In [6]:
# %pip install trl peft

In [26]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import json

# Загружаем токенизатор заранее — он нужен внутри map()
MODEL_FOR_SFT = "Qwen/Qwen2.5-0.5B-Instruct"
sft_tokenizer = AutoTokenizer.from_pretrained(MODEL_FOR_SFT)


def xlam_params_to_json_schema(raw_params):
    """Конвертируем параметры из xlam-формата в стандартную JSON Schema.

    xlam хранит параметры как {param_name: {"description": ..., "type": "str", "default": ...}},
    а JSON Schema ожидает {"type": "object", "properties": {param_name: {"type": "string", ...}}}.
    """
    if not raw_params or not isinstance(raw_params, dict):
        return {"type": "object", "properties": {}}

    type_map = {
        "str": "string", "string": "string",
        "int": "integer", "integer": "integer",
        "float": "number", "number": "number",
        "bool": "boolean", "boolean": "boolean",
        "list": "array", "array": "array",
        "dict": "object", "object": "object",
    }

    properties = {}
    required = []
    for param_name, param_info in raw_params.items():
        if not isinstance(param_info, dict):
            continue
        xlam_type = param_info.get("type", "string")
        prop = {
            "type": type_map.get(xlam_type, "string"),
            "description": param_info.get("description", ""),
        }
        if "enum" in param_info:
            prop["enum"] = param_info["enum"]
        properties[param_name] = prop
        if "default" not in param_info:   # нет default → обязательный параметр
            required.append(param_name)

    schema = {"type": "object", "properties": properties}
    if required:
        schema["required"] = required
    return schema



def convert_xlam_to_text(example):
    """Конвертируем пример из xlam-function-calling-60k в готовый text через apply_chat_template.

    Возвращаем {"text": str} — плоскую строку, чтобы избежать проблем
    с выводом типов Arrow для вложенных структур с разнотипными полями.
    """
    tools_list = json.loads(example["tools"]) if isinstance(example["tools"], str) else example["tools"]
    answers = json.loads(example["answers"]) if isinstance(example["answers"], str) else example["answers"]

    tools_for_template = [
        {
            "type": "function",
            "function": {
                "name": t["name"],
                "description": t.get("description", ""),
                "parameters": xlam_params_to_json_schema(t.get("parameters", {})),
            },
        }
        for t in tools_list
    ]

    messages = [
        {"role": "user", "content": example["query"]},
        {
            "role": "assistant",
            "tool_calls": [
                {
                    "type": "function",
                    "function": {
                        "name": answer["name"],
                        # arguments передаём как dict — apply_chat_template сериализует его сам.
                        # НЕ используем json.dumps: иначе аргументы окажутся двойно-закодированы
                        # ("{\"od\": 0.45}" вместо {"od": 0.45}) и модель выучит неправильный формат.
                        "arguments": answer.get("arguments", {}),
                    },
                }
                for answer in answers
            ],
        },
    ]

    try:
        text = sft_tokenizer.apply_chat_template(
            messages,
            tools=tools_for_template,
            tokenize=False,
            add_generation_prompt=False,
        )
        return {"text": text}
    except Exception:
        return {"text": ""}   # пропускаем проблемные примеры


demo_dataset = xlam_dataset.select(range(1000))
sft_dataset = demo_dataset.map(convert_xlam_to_text, remove_columns=demo_dataset.column_names)
sft_dataset = sft_dataset.filter(lambda x: len(x["text"]) > 0)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [27]:
print(f"Обучающий датасет: {len(sft_dataset)} примеров")
print("\nПример (первые 600 символов):")
print(sft_dataset[0]["text"])

Обучающий датасет: 1000 примеров

Пример (первые 600 символов):
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "live_giveaways_by_type", "description": "Retrieve live giveaways from the GamerPower API based on the specified type.", "parameters": {"type": "object", "properties": {"type": {"type": "string", "description": "The type of giveaways to retrieve (e.g., game, loot, beta)."}}}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Where can I find live giveaways for beta access and games?<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "live_giveaways_

In [28]:
# Загружаем модель (токенизатор уже загружен выше)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

sft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_FOR_SFT,
    quantization_config=bnb_config,
    device_map="auto",
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [29]:
training_args = SFTConfig(
    output_dir="./tool_calling_sft",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    max_length=1024,
    dataset_text_field="text",   # совпадает с ключом, который возвращает convert_xlam_to_text
)

trainer = SFTTrainer(
    model=sft_model,
    args=training_args,
    train_dataset=sft_dataset,
    processing_class=sft_tokenizer,
    peft_config=lora_config,
)

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [30]:
trainer.train()  # Раскомментируйте для запуска обучения
# print("Тренер готов. Для запуска раскомментируйте trainer.train()")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.122505
20,0.640074
30,0.595407
40,0.583319
50,0.564919
60,0.549420


TrainOutput(global_step=63, training_loss=0.6696412979610382, metrics={'train_runtime': 164.1149, 'train_samples_per_second': 6.093, 'train_steps_per_second': 0.384, 'total_flos': 1415097233518080.0, 'train_loss': 0.6696412979610382})

In [31]:
trainer.save_model("./tool_calling_sft_adapter")
sft_tokenizer.save_pretrained("./tool_calling_sft_adapter")
print("Адаптер и токенизатор сохранены в ./tool_calling_sft_adapter")


Адаптер и токенизатор сохранены в ./tool_calling_sft_adapter


### Тестирование: базовая модель vs дообученная

После `trainer.train()` веса LoRA-адаптера обновлены. Базовые веса остались замороженными — это позволяет легко сравнить две модели **без перезагрузки**, просто включая/отключая адаптер.

Оцениваем по трём метрикам:

| Метрика | Описание |
|---|---|
| **Name accuracy** | Правильное имя функции |
| **Args accuracy** | Правильные аргументы (exact match) |
| **Full accuracy** | Имя + аргументы совпадают полностью |

Тестовая выборка — индексы 1000–1099 (не пересекается с обучающей выборкой 0–999).


In [40]:
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(
    MODEL_FOR_SFT,
    quantization_config=bnb_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, "./tool_calling_sft_adapter")




Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [41]:
# model.disable_adapter()

In [42]:
import re
import pandas as pd

pd.set_option('display.max_colwidth', 120)


def make_inference_prompt(example, tokenizer):
    """Промпт для инференса: только user + tools, без ответа ассистента."""
    tools_list = json.loads(example["tools"]) if isinstance(example["tools"], str) else example["tools"]
    tools_for_template = [
        {
            "type": "function",
            "function": {
                "name": t["name"],
                "description": t.get("description", ""),
                "parameters": xlam_params_to_json_schema(t.get("parameters", {})),
            },
        }
        for t in tools_list
    ]
    messages = [{"role": "user", "content": example["query"]}]
    return tokenizer.apply_chat_template(
        messages, tools=tools_for_template, tokenize=False, add_generation_prompt=True
    )


def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """Жадная генерация; возвращает только новые токены."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


def extract_tool_calls(text):
    """Извлекаем все <tool_call>...</tool_call> блоки из ответа Qwen."""
    calls = []
    for m in re.finditer(r'<tool_call>(.*?)</tool_call>', text, re.DOTALL):
        try:
            calls.append(json.loads(m.group(1).strip()))
        except json.JSONDecodeError:
            pass
    return calls


def score_calls(predicted, ground_truth):
    """Возвращает (name_ok, args_ok, full_ok).

    Сравниваем отсортированные списки вызовов, чтобы не штрафовать
    за перестановку параллельных вызовов.
    """
    if len(predicted) != len(ground_truth):
        return False, False, False

    # Сортируем по имени, чтобы порядок параллельных вызовов не влиял
    pred_s = sorted(predicted, key=lambda x: x.get("name", ""))
    gt_s   = sorted(ground_truth, key=lambda x: x.get("name", ""))

    name_ok = all(p.get("name") == g.get("name") for p, g in zip(pred_s, gt_s))

    # Аргументы сравниваем как dict (нечувствительно к порядку ключей в JSON)
    args_ok = all(
        p.get("arguments", {}) == g.get("arguments", {})
        for p, g in zip(pred_s, gt_s)
    )
    return name_ok, args_ok, name_ok and args_ok


print("Вспомогательные функции определены")


Вспомогательные функции определены


In [43]:
# ── Один пример: наглядное сравнение ──────────────────────────────────────
example = xlam_dataset[1001]   # первый пример из тестовой выборки

ground_truth = json.loads(example["answers"])
prompt = make_inference_prompt(example, sft_tokenizer)

print("Запрос:", example["query"])
print("Эталон:", json.dumps(ground_truth, ensure_ascii=False))
print()

# Базовая модель (адаптер выключен)
with model.disable_adapter():
    base_output = generate_response(model, sft_tokenizer, prompt)

# sft_model.enable_adapter_layers()

# Дообученная модель (адаптер включён)
ft_output = generate_response(model, sft_tokenizer, prompt)

print("=== Базовая модель ===")
print(base_output)
print()
print("=== Дообученная модель ===")
print(ft_output)
print()

# Парсим и оцениваем
for label, output in [("Base", base_output), ("Fine-tuned", ft_output)]:
    calls = extract_tool_calls(output)
    name_ok, args_ok, full_ok = score_calls(calls, ground_truth)
    print(f"{label:12s} | parsed calls: {len(calls)} | name: {name_ok} | args: {args_ok} | full: {full_ok}")


Запрос: If an object starts with an initial velocity of 5 m/s and accelerates at 2 m/s² for 10 seconds, what is its final velocity?
Эталон: [{"name": "final_velocity", "arguments": {"initial_velocity": 5, "acceleration": 2, "time": 10}}]



/venv/main/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


=== Базовая модель ===
<tool_call>
{"name": "final_velocity", "arguments": {"initial_velocity": 5, "acceleration": 2, "time": 10}}
</tool_call>

=== Дообученная модель ===
<tool_call>
{"name": "final_velocity", "arguments": {"initial_velocity": 5, "acceleration": 2, "time": 10}}
</tool_call>

Base         | parsed calls: 1 | name: True | args: True | full: True
Fine-tuned   | parsed calls: 1 | name: True | args: True | full: True


На одном примере разницы не видно, давайте прогоним хотя бы 100!

In [45]:
# ── Оценка на тестовой выборке (100 примеров, индексы 1000–1099) ───────────
from tqdm.auto import tqdm

TEST_SIZE = 100
test_examples = [xlam_dataset[i] for i in range(1000, 1000 + TEST_SIZE)]

results = []

for example in tqdm(test_examples, desc="Evaluating"):
    gt_calls = json.loads(example["answers"])
    prompt = make_inference_prompt(example, sft_tokenizer)

    # Базовая модель
    with model.disable_adapter():
        base_out = generate_response(model, sft_tokenizer, prompt,  max_new_tokens=256)
    
    # Дообученная модель
    ft_out = generate_response(model, sft_tokenizer, prompt, max_new_tokens=256)

    base_calls = extract_tool_calls(base_out)
    ft_calls   = extract_tool_calls(ft_out)

    b_name, b_args, b_full = score_calls(base_calls, gt_calls)
    f_name, f_args, f_full = score_calls(ft_calls,   gt_calls)

    results.append({
        "query":          example["query"][:60] + "...",
        "n_gt_calls":     len(gt_calls),
        "base_parsed":    len(base_calls),
        "ft_parsed":      len(ft_calls),
        "base_name":      b_name,
        "ft_name":        f_name,
        "base_args":      b_args,
        "ft_args":        f_args,
        "base_full":      b_full,
        "ft_full":        f_full,
    })

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

In [46]:
df = pd.DataFrame(results)

print(f"\n{'Метрика':<25} {'Базовая':>10} {'Fine-tuned':>12} {'Δ':>8}")
print("-" * 58)
for col, label in [
    ("name", "Name accuracy"),
    ("args", "Args accuracy"),
    ("full", "Full accuracy"),
]:
    base_acc = df[f"base_{col}"].mean()
    ft_acc   = df[f"ft_{col}"].mean()
    delta    = ft_acc - base_acc
    print(f"{label:<25} {base_acc:>9.1%} {ft_acc:>11.1%} {delta:>+8.1%}")

parse_rate_base = (df["base_parsed"] == df["n_gt_calls"]).mean()
parse_rate_ft   = (df["ft_parsed"]   == df["n_gt_calls"]).mean()
print(f"{'Parse rate (count match)':<25} {parse_rate_base:>9.1%} {parse_rate_ft:>11.1%} {parse_rate_ft - parse_rate_base:>+8.1%}")


Метрика                      Базовая   Fine-tuned        Δ
----------------------------------------------------------
Name accuracy                 42.0%       88.0%   +46.0%
Args accuracy                 24.0%       49.0%   +25.0%
Full accuracy                 24.0%       49.0%   +25.0%
Parse rate (count match)      44.0%       94.0%   +50.0%


###### ---

# Structured Outputs

### Проблема

Представьте, что вы пишете систему, которая должна:
1. Принять запрос пользователя
2. Попросить LLM распарсить его в структурированный формат
3. Передать структуру в следующую систему (БД, API, другой компонент)

```python
# Хотим получить:
{
    "intent": "book_flight",
    "destination": "Moscow",
    "date": "2026-03-15",
    "passengers": 2
}

# Модель может сгенерировать:
# - Правильный JSON ✓
# - JSON с лишним текстом: "Sure! Here is the JSON: {...}" ✗
# - Невалидный JSON: {intent: book_flight, ...} ✗  
# - Неправильные типы: {"passengers": "two"} ✗
# - Отсутствующие поля ✗
```



---

###  Constrained Decoding

### Как работает

На каждом шаге генерации модель выдаёт логиты по всему словарю (обычно 32k–128k токенов). Обычная генерация: `argmax` или `sampling` по всем токенам.

Constrained decoding добавляет **маску**: перед сэмплингом зануляет логиты токенов, которые не могут быть следующими по грамматике/схеме.

```
Генерируем: {"name": "
Словарь: ["Alice", "Bob", "  ", "}", "123", ...]

Без маски:   все токены → может получиться {"name": 123}
С маской:    только строки → гарантированно {"name": "Alice"}
```

### Конечные автоматы (FSM) для регулярных выражений

Для регулярных выражений грамматика компилируется в **детерминированный конечный автомат (DFA)**:
- Состояние DFA соответствует «где мы сейчас в парсинге»
- Переходы — допустимые символы
- Маска токенов строится из допустимых переходов из текущего состояния

```
Regex: [0-9]+\.?[0-9]*

Состояние 0: ожидаем цифру → допустимы ["0".."9"]
Состояние 1: цифры идут → допустимы ["0".."9", "."]
Состояние 2: после точки → допустимы ["0".."9"]
```

### JSON Schema через Context-Free Grammar (CFG)

JSON Schema нельзя описать регулярным выражением (вложенные объекты — рекурсивная структура). Поэтому используют **контекстно-свободные грамматики (CFG)**.

Библиотеки реализуют **incremental parsing**: на каждом шаге определяют множество токенов, которые не нарушат грамматику.

## Библиотека outlines для constrained decoding

In [47]:
%pip install -q outlines

Note: you may need to restart the kernel to use updated packages.


In [54]:
import outlines
from outlines.types import Regex, CFG
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
outline_model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"),
    AutoTokenizer.from_pretrained(MODEL_NAME),
)

print("Модель загружена через Outlines")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Модель загружена через Outlines


### Генерация по регулярному выражению

In [55]:
number_generator = outlines.Generator(outline_model, Regex(r"-?[0-9]+(\.[0-9]+)?"))

prompt = "What is 15 * 7? Just give the number:"
result = number_generator(prompt)
print(f"Ответ: '{result}'")
print(f"Тип: {type(result)}")
print(f"Можно сконвертировать: {float(result)}")


/venv/main/lib/python3.12/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=63) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Ответ: '90'
Тип: <class 'str'>
Можно сконвертировать: 90.0


In [56]:
# Генерация даты в формате YYYY-MM-DD
from datetime import datetime

date_generator = outlines.Generator(
    outline_model,
    Regex(r"(19|20)[0-9]{2}-(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01])")
)

prompt = "What is today's date? (In format YYYY-MM-DD):"
result = date_generator(prompt)
print(f"Дата: {result}")

# Гарантированно валидный формат!
try:
    parsed = datetime.strptime(result, "%Y-%m-%d")
    print(f"Парсится как datetime: {parsed}")
except Exception as e:
    print(f"Ошибка парсинга — но этого не должно происходить с Outlines: {e}")


/venv/main/lib/python3.12/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=62) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Дата: 2023-10-07
Парсится как datetime: 2023-10-07 00:00:00


In [57]:
# Выбор из набора значений (Enum)
from enum import Enum

class Sentiment(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL  = "neutral"


sentiment_generator = outlines.Generator(outline_model, Sentiment)

texts = [
    "I love this product! It's amazing!",
    "This is terrible, I want a refund.",
    "The product arrived on time.",
]

for text in texts:
    prompt = f"Classify the sentiment of this text: '{text}'\nSentiment:"
    sentiment = sentiment_generator(prompt)
    print(f"  '{text[:40]}...' → {sentiment}")


  'I love this product! It's amazing!...' → positive
  'This is terrible, I want a refund....' → negative
  'The product arrived on time....' → positive


/venv/main/lib/python3.12/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=70) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/venv/main/lib/python3.12/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=67) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


### Генерация структурированного JSON через Pydantic

Важный инструмент при работе с constrained decoding и structured outputs в целом - это pydantic. Это уже базовая составляющая питона в целом, которая используется для аннотации типов а также автоматической проверки следованию заданному формату. Pydantic позволяет генерировать схемы через обычный питоновский код. Outline и некоторые другие библиотеки используют pydantic для автотической проверки результатов LLM. Такую генерацию потом можно сразу же распарсить с использованием того же pydantic и получить питоновский объект, а не сырую строку!

In [58]:
%pip install -q pydantic

Note: you may need to restart the kernel to use updated packages.


In [67]:
from pydantic import BaseModel, Field
from typing import List, Optional
from pydantic import ValidationError

class FlightBooking(BaseModel):
    intent:     str          = Field(description="User intent") # тут описание это просто документация, но в tool calls это пойдет в промпт!
    origin:     str          = Field(description="Departure city")
    destination: str         = Field(description="Destination city")
    date:       str          = Field(description="Travel date in YYYY-MM-DD format")
    passengers: int          = Field(ge=1, le=10, description="Number of passengers")
    class_type: Optional[str]= Field(default="economy", description="Ticket class")


flight_generator = outlines.Generator(outline_model, FlightBooking, )

prompt = """Extract flight booking information from this request:
\"I need two economy tickets from Moscow to Paris on March 15, 2025.\"
"""

In [68]:
result = flight_generator(prompt, max_new_tokens=100)
print(f"Тип результата: {type(result)}")
print(f"Результат: {result}")

try:
    result_data = FlightBooking.model_validate_json(result)
    print(f"Пассажиры: {result_data.passengers} (тип: {type(result_data.passengers).__name__})")
except ValidationError:
    print("неправильный или неполный json")

Тип результата: <class 'str'>
Результат: {"intent": "I need two economy tickets from Moscow to Paris on March 15, 2025.", "origin": "Moscow", "destination": "Paris", "date": "March 15, 2025", "passengers": 2}
Пассажиры: 2 (тип: int)


In [74]:
# Более сложная схема: вложенные объекты
class Address(BaseModel):
    street:  str
    city:    str
    country: str

class Person(BaseModel):
    name:       str
    age:        int       = Field(ge=0, le=150)
    occupation: str
    address:    Address
    hobbies:    List[str]


person_generator = outlines.Generator(outline_model, Person)

prompt = """Generate a fictional person profile:
Name: Anna Ivanova, 28 years old, software engineer from Moscow, Russia.
She likes hiking and photography."""

person = person_generator(prompt, max_new_tokens=100)
print("Сгенерированный профиль:")
print(person)

Сгенерированный профиль:
{"name": "Anna Ivanova", "age": 28, "occupation": "software engineer", "address": {"street": "Moscow Street 123", "city": "Moscow", "country": "Russia"}, "hobbies": ["hiking", "photography"]}


In [75]:
Person.model_validate_json(person)

Person(name='Anna Ivanova', age=28, occupation='software engineer', address=Address(street='Moscow Street 123', city='Moscow', country='Russia'), hobbies=['hiking', 'photography'])

### Использование произвольной JSON Schema

In [78]:
import json

# Можно передать JSON Schema напрямую (без Pydantic) через outlines.json_schema()
# Принимает JSON-схему как строку
schema = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "text":  {"type": "string"},
                    "label": {"type": "string", "enum": ["PERSON", "ORG", "LOC", "DATE"]},
                    "start": {"type": "integer"},
                    "end":   {"type": "integer"}
                },
                "required": ["text", "label"]
            }
        }
    },
    "required": ["entities"]
}

ner_generator = outlines.Generator(outline_model, outlines.json_schema(json.dumps(schema)))

text = "Barack Obama visited Moscow in 2014 with Google CEO Sundar Pichai."
prompt = f"""Extract named entities from this text and classify them as PERSON, ORG, LOC, or DATE:
Text: \"{text}\"
JSON:"""

entities = ner_generator(prompt, max_new_tokens=200)
print("Извлечённые сущности:")
print(json.dumps(entities, indent=2, ensure_ascii=False))

Извлечённые сущности:
"{\"entities\":[{\"text\":\"Barack Obama\",\"label\":\"PERSON\"},{\"text\":\"Moscow\",\"label\":\"LOC\"},{\"text\":\"Google CEO Sundar Pichai\",\"label\":\"ORG\"}]}"


### Использование контекстно-свободной грамматики (CFG)

Есть инструменты, которые позволяют писать полноценные грамматики и использовать их для контроля генерации. Самая известная это lark - https://github.com/lark-parser/lark

In [101]:
# %pip install llguidance

In [83]:
# Можно задать грамматику в формате LARK через CFG из outlines.types
arithmetic_grammar = """
    ?start: expr
    ?expr: term ("+" term | "-" term)*
    ?term: factor ("*" factor | "/" factor)*
    ?factor: NUMBER | "(" expr ")"
    NUMBER: /[0-9]+/
    %ignore " "
"""

# CFG гарантирует синтаксически корректное арифметическое выражение
arithmetic_generator = outlines.Generator(outline_model, CFG(arithmetic_grammar))

prompt = "Write a simple arithmetic expression with two operations:"
expr = arithmetic_generator(prompt, max_new_tokens=20)
print(f"Выражение: {expr}")
print(f"Результат: {eval(expr)}")


Выражение: 1/2 + 3*4 - 5 / 2 * 6 + 7
Результат: 4.5


### Как эти техники связаны с RL/GRPO?

Outline позволяет форсить правильный output формат и это очень удобно и позволяет получить хоть какой-то детерминизм от LLM. Однако хотелось бы применять такие инструменты только как крайнюю меру. Гораздо лучше когда модель сама справляется со следованием формату. Для этого модель нужно обучать. Мы уже посмотрели выше на дообучение SFT способом, когда мы хотим научить модель работать с tools. С форматом можно делать что-то такое же, но RL тут подходит гораздо лучше. 

Вспомним из прошлого семинара: в GRPO мы использовали `format_reward` — проверку, что модель сгенерировала `#### <число>`. Это простейший случай structured output как reward-сигнала. Но применить это к настойящему structured output совсем не сложно. Инструменты вроде pydantic предоставляют детерменистическую валидацию, которую можно использовать как reward (проходит валидацию или нет, нужный tool или нет, корректный формат аргументов или нет и т.п). И такое обучение можно очень хорошо масштабировать потому что можно программно генерировать обучающие примеры. 



**Например:**

```python
def tool_call_format_reward(completions, expected_tool=None, **kwargs):
    """Reward за корректный JSON-вызов инструмента."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        
        # Проверяем формат
        tool_call = parse_tool_call(text)
        if tool_call is None:
            rewards.append(0.0)
            continue
        
        reward = 0.5  # Базовый reward за валидный JSON
        
        # Бонус за правильное имя инструмента
        if expected_tool and tool_call.get("name") == expected_tool:
            reward += 0.5
        
        rewards.append(reward)
    
    return rewards
```

Также можно пойти еще дальше. Правильный формат еще не гарантирует, что данные будут правильными с точки зрения задачи. И можно применять тот же подход, что и в Deepseek только в более глобальном формате. Можно брать supervised датасеты для разных задач, задавать формат для модели и затем проверять совпадает ответ с правильным или нет. Так модель будет учиться и следовать разным форматам и давать правильные ответы. 